# 05 - Ocean Surface Temperature On A Curvilinear Grid

Create a CMIP7 monthly sea surface temperature file (`tos_tavg-u-hxy-sea`) using `cmor.grid` with two-dimensional latitude and longitude coordinates.

In [ ]:
from pathlib import Path
import json
import shutil

import cmor
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "notebooks" / "NOTEBOOKS.md").exists() or (candidate / ".git").exists():
        REPO_ROOT = candidate
        break
else:
    REPO_ROOT = start

TABLES_REPO = REPO_ROOT / "cmip7-cmor-tables"
TABLES_DIR = TABLES_REPO / "tables"
if not TABLES_DIR.exists():
    raise FileNotFoundError(f"CMIP7 tables directory not found: {TABLES_DIR}")

CV_FILE_FROM_TABLES_DIR = "../tables-cvs/cmor-cvs.json"

print(f"Using CMIP7 tables from {TABLES_REPO}")


In [ ]:
run_dir = REPO_ROOT / "notebooks" / "output" / "05_curvilinear_ocean_grid"
if run_dir.exists():
    shutil.rmtree(run_dir)
output_dir = run_dir / "cmor_output"
output_dir.mkdir(parents=True)

DATASET_INFO = {
    "_AXIS_ENTRY_FILE": "CMIP7_coordinate.json",
    "_FORMULA_VAR_FILE": "CMIP7_formula_terms.json",
    "_cmip7_option": 1,
    "_controlled_vocabulary_file": CV_FILE_FROM_TABLES_DIR,
    "activity_id": "CMIP",
    "calendar": "360_day",
    "drs_specs": "MIP-DRS7",
    "experiment_id": "amip",
    "forcing_index": "f3",
    "frequency": "mon",
    "grid_label": "g999",
    "host_collection": "CMIP7",
    "initialization_index": "i1",
    "institution_id": "CCCma",
    "license_id": "CC-BY-4.0",
    "mip_era": "CMIP7",
    "nominal_resolution": "100 km",
    "outpath": str(output_dir),
    "physics_index": "p1",
    "realization_index": "r9",
    "region": "glb",
    "source_id": "DUMMY-MODEL",
    "tracking_prefix": "hdl:21.14107",
}
input_json = run_dir / "input.json"
input_json.write_text(json.dumps(DATASET_INFO, indent=2, sort_keys=True))

ny, nx = 18, 36
y = np.arange(ny, dtype="f8")
x = np.arange(nx, dtype="f8")
lat_centers = -85.0 + 10.0 * np.arange(ny, dtype="f8")
lon_centers = 5.0 + 10.0 * np.arange(nx, dtype="f8")
lon_grid, lat_grid = np.meshgrid(lon_centers, lat_centers)
latitude = np.clip(lat_grid + 1.5 * np.sin(np.deg2rad(lon_grid)) * np.cos(np.deg2rad(lat_grid)), -90.0, 90.0)
longitude = (lon_grid + 1.5 * np.sin(np.deg2rad(lat_grid))) % 360.0

lat_edges = -90.0 + 10.0 * np.arange(ny + 1, dtype="f8")
lon_edges = 10.0 * np.arange(nx + 1, dtype="f8")
latitude_vertices = np.empty((ny, nx, 4), dtype="f8")
longitude_vertices = np.empty((ny, nx, 4), dtype="f8")
for j in range(ny):
    for i in range(nx):
        corners = [
            (lat_edges[j], lon_edges[i]),
            (lat_edges[j], lon_edges[i + 1]),
            (lat_edges[j + 1], lon_edges[i + 1]),
            (lat_edges[j + 1], lon_edges[i]),
        ]
        for vertex, (lat_value, lon_value) in enumerate(corners):
            latitude_vertices[j, i, vertex] = np.clip(
                lat_value + 1.5 * np.sin(np.deg2rad(lon_value)) * np.cos(np.deg2rad(lat_value)),
                -90.0,
                90.0,
            )
            longitude_vertices[j, i, vertex] = (lon_value + 1.5 * np.sin(np.deg2rad(lat_value))) % 360.0

time = 15.0 + 30.0 * np.arange(2, dtype="f8")
time_bnds = 30.0 * np.arange(3, dtype="f8")
time_units = "days since 2018-01-01"

seasonal_cycle = np.array([0.0, 0.8], dtype="f4")[:, None, None]
tos = (
    15.0
    - 25.0 * np.sin(np.deg2rad(latitude))[None, :, :] ** 2
    + 1.5 * np.cos(np.deg2rad(longitude))[None, :, :]
    + seasonal_cycle
).astype("f4")

tos.shape


In [ ]:
cmor.setup(
    inpath=str(TABLES_DIR),
    netcdf_file_action=cmor.CMOR_REPLACE,
    logfile=str(run_dir / "cmor.log"),
)
cmor.dataset_json(str(input_json))

grid_table = cmor.load_table("CMIP7_grids.json")
ocean_table = cmor.load_table("CMIP7_ocean.json")

cmor.set_table(grid_table)
y_id = cmor.axis("y", coord_vals=y, units="m")
x_id = cmor.axis("x", coord_vals=x, units="m")
grid_id = cmor.grid(
    axis_ids=[y_id, x_id],
    latitude=latitude,
    longitude=longitude,
    latitude_vertices=latitude_vertices,
    longitude_vertices=longitude_vertices,
)

cmor.set_table(ocean_table)
time_id = cmor.axis("time", coord_vals=time, cell_bounds=time_bnds, units=time_units)

variable_name = "tos_tavg-u-hxy-sea"
tos_id = cmor.variable(variable_name, "degC", [time_id, grid_id])
compound_name = ".".join(["ocean"] + variable_name.split("_") + ["mon", "glb"])

with open(TABLES_DIR / "CMIP7_cell_measures.json") as handle:
    cell_measure = json.load(handle)["cell_measures"].get(compound_name)
if cell_measure:
    cmor.set_variable_attribute(tos_id, "cell_measures", "c", cell_measure)

with open(TABLES_DIR / "CMIP7_long_name_overrides.json") as handle:
    long_name = json.load(handle)["long_name_overrides"].get(compound_name)
if long_name:
    cmor.set_variable_attribute(tos_id, "long_name", "c", long_name)

cmor.write(tos_id, tos)
netcdf_path = Path(cmor.close(tos_id, file_name=True))
cmor.close()

netcdf_path


In [ ]:
with xr.open_dataset(netcdf_path, decode_times=False) as opened:
    ds = opened.load()
ds


In [ ]:
print(ds[["tos", "latitude", "longitude"]])
print("coordinates on tos:", ds["tos"].attrs.get("coordinates"))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
mesh = ax.pcolormesh(
    ds["longitude"],
    ds["latitude"],
    ds["tos"].isel(time=0),
    shading="auto",
    cmap="coolwarm",
)
fig.colorbar(mesh, ax=ax, label="degC")
ax.set_title("CMIP7 tos on a curvilinear grid, first month")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(0, 360)
ax.set_ylim(-90, 90)
ax.grid(True, alpha=0.25)
plt.show()
